<style type="text/css">
.main-container {
  max-width: 100% !important;
  margin: auto;
}
</style>


# FED Rate Expectations Over Time


```r
filename_ = file.info(list.files("FedMeetingHistory",path = "Data",full.names = T)) %>% filter(mtime==max(mtime)) %>% row.names(.)
full_file = read.csv(file = filename_,header = FALSE)
for (col_num in seq(2,ncol(full_file))) {
  full_file[1,col_num] = ifelse(""==(full_file[1,col_num]),full_file[1,col_num-1],full_file[1,col_num])
}
cols_to_loop = which(full_file %>% slice(1) %>% as.character() %like% "History for")
col_names = full_file %>% slice(1) %>% as.character()

rate_probs = do.call("rbind",lapply(seq(2,length(cols_to_loop)),function(cols_) {
  temp_df = full_file %>% 
    dplyr::select(1,c(cols_to_loop[cols_-1]:(cols_to_loop[cols_]-1))) %>% 
    slice(c(2:nrow(.)))
  colnames(temp_df) = temp_df %>% slice(c(1)) %>% as.character()
  temp_df = temp_df %>% 
    tail(-1) %>% 
    gather(Rate,Val,c(2:ncol(.))) %>% 
    dplyr::mutate(Meeting=col_names[cols_to_loop[cols_]])
  temp_df
    
})) %>% 
  dplyr::mutate(Val=as.numeric(Val),
                Rate=factor(Rate),
                Date=mdy(Date)) %>% 
  dplyr::mutate(Meeting=gsub("History for ","",Meeting)) %>% 
  dplyr::mutate(Meeting=gsub(" Fed meeting","",Meeting)) %>% 
  dplyr::mutate(Meeting=as.Date(lubridate::parse_date_time(Meeting,"%d %b %Y"))) %>% 
  dplyr::mutate(Rate=gsub("\\)","",Rate)) %>% 
  dplyr::mutate(Rate=stringr::str_extract(Rate,"(?<=-).*")) %>% 
  dplyr::mutate(Rate=as.numeric(Rate)/100)

rate_probs %>% 
  group_by(floor_date(Date,unit = "month")) %>% 
  filter(Date==min(Date,na.rm=T)) %>% 
  dplyr::mutate(Date=factor(format.Date(Date,"%Y %b"))) %>% 
  group_by(Date,Meeting) %>% 
  filter(Val==max(Val,na.rm=T)) %>% 
  ggplot(.,aes(x=Meeting,y=Rate,color=Date)) +
  geom_line() +
  geom_point() +
  scale_x_date(date_breaks = "1 month",date_labels = "%b")
```


```r
# DFF
df_dff = df_yieldcurve %>% 
  filter(series_id=="DFF")

# Rate Probabilities
filename_ = file.info(list.files("FedMeetingHistory",path = "Data",full.names = T)) %>% filter(mtime==max(mtime)) %>% row.names(.)
full_file = read.csv(file = filename_,header = FALSE)
# for (col_num in seq(2,ncol(full_file))) {
#   full_file[1,col_num] = ifelse(""==(full_file[1,col_num]),full_file[1,col_num-1],full_file[1,col_num])
# }
cols_to_loop = which(full_file %>% slice(1) %>% as.character() %like% "History for")
col_names = full_file %>% slice(1) %>% as.character()

rate_probs = do.call("rbind",lapply(seq(2,length(cols_to_loop)),function(cols_) {
  temp_df = full_file %>% 
    dplyr::select(1,c(cols_to_loop[cols_-1]:(cols_to_loop[cols_]-1))) %>% 
    slice(c(2:nrow(.)))
  colnames(temp_df) = temp_df %>% slice(c(1)) %>% as.character()
  temp_df = temp_df %>% 
    tail(-1) %>% 
    gather(Rate,Val,c(2:ncol(.))) %>% 
    dplyr::mutate(Meeting=col_names[cols_to_loop[cols_]]) %>% 
  dplyr::mutate(Val=as.numeric(Val),
                Rate=factor(Rate),
                Date=mdy(Date)) %>% 
  dplyr::mutate(Meeting=gsub("History for ","",Meeting)) %>% 
  dplyr::mutate(Meeting=gsub(" Fed meeting","",Meeting)) %>% 
  dplyr::mutate(Meeting=as.Date(parse_date_time(Meeting,"%d %b %Y"))) %>% 
  dplyr::mutate(Rate=gsub("\\)","",Rate)) %>% 
  dplyr::mutate(Rate=stringr::str_extract(Rate,"(?<=-).*")) %>% 
  dplyr::mutate(Rate=as.numeric(Rate)/100) %>% 
  dplyr::mutate(floor_date_=floor_date(Date,unit = "month")) %>% 
  group_by(floor_date_) %>% 
  filter(Date==min(Date,na.rm=T)) %>% 
  group_by(Date,Meeting) %>% 
  filter(Val==max(Val,na.rm=T)) %>% 
  ungroup() 
  
  df_dff_bind =   df_dff %>% 
    dplyr::select(date,value) %>% 
    inner_join( temp_df %>% dplyr::select(Date) %>% dplyr::rename(date=Date)) %>% 
    dplyr::mutate(Meeting=date) %>% 
    dplyr::rename(Date=date,
                  Rate=value)
  
  temp_df %>% 
    bind_rows( df_dff_bind ) %>% 
    dplyr::mutate(Date=factor(format.Date(Date,format="%Y %b")))
    
}))


df_dff %>% 
  filter(date>=dmy("01-01-2022")) %>% 
  ggplot(.,aes(x=date,y=value)) +
  geom_line() +
  geom_line(data=rate_probs,aes(x=Meeting,y=Rate,color=Date)) +
  geom_point(data=rate_probs,aes(x=Meeting,y=Rate,color=Date)) +
  # geom_point() +
  scale_x_date(date_breaks = "1 month",date_labels = "%b") +
  scale_y_continuous(breaks = seq(0,6,by=0.25))
```


Terminal rate


```r
# DFF
df_dff = df_yieldcurve %>% 
  filter(series_id=="DFF")

# Rate Probabilities
filename_ = file.info(list.files("FedMeetingHistory",path = "Data",full.names = T)) %>% filter(mtime==max(mtime)) %>% row.names(.)
full_file = read.csv(file = filename_,header = FALSE)
# for (col_num in seq(2,ncol(full_file))) {
#   full_file[1,col_num] = ifelse(""==(full_file[1,col_num]),full_file[1,col_num-1],full_file[1,col_num])
# }
cols_to_loop = which(full_file %>% slice(1) %>% as.character() %like% "History for")
col_names = full_file %>% slice(1) %>% as.character()

rate_probs = do.call("rbind",lapply(seq(2,length(cols_to_loop)),function(cols_) {
  temp_df = full_file %>% 
    dplyr::select(1,c(cols_to_loop[cols_-1]:(cols_to_loop[cols_]-1))) %>% 
    slice(c(2:nrow(.)))
  colnames(temp_df) = temp_df %>% slice(c(1)) %>% as.character()
  temp_df = temp_df %>% 
    tail(-1) %>% 
    gather(Rate,Val,c(2:ncol(.))) %>% 
    dplyr::mutate(Meeting=col_names[cols_to_loop[cols_]]) %>% 
  dplyr::mutate(Val=as.numeric(Val),
                Rate=factor(Rate),
                Date=mdy(Date)) %>% 
  dplyr::mutate(Meeting=gsub("History for ","",Meeting)) %>% 
  dplyr::mutate(Meeting=gsub(" Fed meeting","",Meeting)) %>% 
  dplyr::mutate(Meeting=as.Date(parse_date_time(Meeting,"%d %b %Y"))) %>% 
  dplyr::mutate(Rate=gsub("\\)","",Rate)) %>% 
  dplyr::mutate(Rate=stringr::str_extract(Rate,"(?<=-).*")) %>% 
  dplyr::mutate(Rate=as.numeric(Rate)/100-0.25) %>% 
  # dplyr::mutate(floor_date_=floor_date(Date,unit = "month")) %>% 
  # group_by(floor_date_) %>% 
  # filter(Date==min(Date,na.rm=T)) %>% 
  group_by(Date,Meeting) %>% 
  filter(Val==max(Val,na.rm=T)) %>% 
  ungroup() 
  
  df_dff_bind =   df_dff %>% 
    dplyr::select(date,value) %>% 
    inner_join( temp_df %>% dplyr::select(Date) %>% dplyr::rename(date=Date)) %>% 
    dplyr::mutate(Meeting=date) %>% 
    dplyr::rename(Date=date,
                  Rate=value)
  
  temp_df %>% 
    bind_rows( df_dff_bind )
    
})) %>% 
    group_by(Date) %>% 
    dplyr::summarize(Rate=max(Rate,na.rm=T)) %>% 
    ungroup()


df_dff %>% 
  filter(date>=dmy("01-01-2022")) %>% 
  ggplot(.,aes(x=date,y=value)) +
  geom_line() +
  geom_line(data=rate_probs,aes(x=Date,y=Rate)) +
  geom_point(data=rate_probs,aes(x=Date,y=Rate)) +
  # geom_point() +
  scale_x_date(date_breaks = "1 month",date_labels = "%Y %b") +
  scale_y_continuous(breaks = seq(0,6,by=0.25)) +
  labs(x=NULL,y=NULL) +
  theme_bw() +
  theme(axis.text.x=element_text(angle=90,hjust=1))
```


Terminal rate expected data


```r
# DFF
df_dff = df_yieldcurve %>% 
  filter(series_id=="DFF")

# Rate Probabilities
filename_ = file.info(list.files("FedMeetingHistory",path = "Data",full.names = T)) %>% filter(mtime==max(mtime)) %>% row.names(.)
full_file = read.csv(file = filename_,header = FALSE)
# for (col_num in seq(2,ncol(full_file))) {
#   full_file[1,col_num] = ifelse(""==(full_file[1,col_num]),full_file[1,col_num-1],full_file[1,col_num])
# }
cols_to_loop = which(full_file %>% slice(1) %>% as.character() %like% "History for")
col_names = full_file %>% slice(1) %>% as.character()

rate_probs = do.call("rbind",lapply(seq(2,length(cols_to_loop)),function(cols_) {
  temp_df = full_file %>% 
    dplyr::select(1,c(cols_to_loop[cols_-1]:(cols_to_loop[cols_]-1))) %>% 
    slice(c(2:nrow(.)))
  colnames(temp_df) = temp_df %>% slice(c(1)) %>% as.character()
  temp_df = temp_df %>% 
    tail(-1) %>% 
    gather(Rate,Val,c(2:ncol(.))) %>% 
    dplyr::mutate(Meeting=col_names[cols_to_loop[cols_]]) %>% 
  dplyr::mutate(Val=as.numeric(Val),
                Rate=factor(Rate),
                Date=mdy(Date)) %>% 
  dplyr::mutate(Meeting=gsub("History for ","",Meeting)) %>% 
  dplyr::mutate(Meeting=gsub(" Fed meeting","",Meeting)) %>% 
  dplyr::mutate(Meeting=as.Date(parse_date_time(Meeting,"%d %b %Y"))) %>% 
  dplyr::mutate(Rate=gsub("\\)","",Rate)) %>% 
  dplyr::mutate(Rate=stringr::str_extract(Rate,"(?<=-).*")) %>% 
  dplyr::mutate(Rate=as.numeric(Rate)/100-0.25) %>% 
  # dplyr::mutate(floor_date_=floor_date(Date,unit = "month")) %>% 
  # group_by(floor_date_) %>% 
  # filter(Date==min(Date,na.rm=T)) %>% 
  group_by(Date,Meeting) %>% 
  filter(Val==max(Val,na.rm=T)) %>% 
  ungroup() 
  
  df_dff_bind =   df_dff %>% 
    dplyr::select(date,value) %>% 
    inner_join( temp_df %>% dplyr::select(Date) %>% dplyr::rename(date=Date)) %>% 
    dplyr::mutate(Meeting=date) %>% 
    dplyr::rename(Date=date,
                  Rate=value)
  
  temp_df %>% 
    bind_rows( df_dff_bind )
    
})) %>% 
    group_by(Date) %>% 
    dplyr::filter(Rate==max(Rate,na.rm=T)) %>% 
    ungroup() %>% 
    dplyr::select(Date,Meeting,Rate) #%>% 
    # dplyr::mutate(Rate=factor(Rate))


rate_probs %>% 
  # filter(date>=dmy("01-01-2022")) %>% 
  ggplot(.,aes(x=Meeting,y=Rate,color=Date)) +
  geom_point() +
  # geom_point() +
  # scale_y_date(date_breaks = "1 month",date_labels = "%Y %b") +
  scale_x_date(date_breaks = "1 month",date_labels = "%Y %b") +
  labs(x=NULL,y=NULL) +
  theme_bw() +
  theme(axis.text.x=element_text(angle=45,hjust=1))
```


Terminal rate expectations


In [ ]:
#| echo: false
#| warning: false
options(repr.plot.width = 10, repr.plot.height = 7, repr.plot.res = 120)
# DFF
df_dff = df_yieldcurve %>% 
  filter(series_id=="DFF") %>% 
  tidyr::complete(date=seq.Date(from=max(date,na.rm=T),to=Sys.Date(),by="1 day")) %>% 
  tidyr::fill(value,.direction = "down")  %>% 
  tidyr::fill(series_id,.direction = "down")


```r
# DFF
df_dff = df_yieldcurve %>% 
  filter(series_id=="DFF")

# Rate Probabilities
filename_ = file.info(list.files("FedMeetingHistory",path = "Data",full.names = T)) %>% filter(mtime==max(mtime)) %>% row.names(.)
full_file = read.csv(file = filename_,header = FALSE)
# for (col_num in seq(2,ncol(full_file))) {
#   full_file[1,col_num] = ifelse(""==(full_file[1,col_num]),full_file[1,col_num-1],full_file[1,col_num])
# }
cols_to_loop = which(full_file %>% slice(1) %>% as.character() %like% "History for")
col_names = full_file %>% slice(1) %>% as.character()

rate_probs = do.call("rbind",lapply(seq(2,length(cols_to_loop)),function(cols_) {
  temp_df = full_file %>% 
    dplyr::select(1,c(cols_to_loop[cols_-1]:(cols_to_loop[cols_]-1))) %>% 
    slice(c(2:nrow(.)))
  colnames(temp_df) = temp_df %>% slice(c(1)) %>% as.character()
  temp_df = temp_df %>% 
    tail(-1) %>% 
    gather(Rate,Val,c(2:ncol(.))) %>% 
    dplyr::mutate(Meeting=col_names[cols_to_loop[cols_]]) %>% 
  dplyr::mutate(Val=as.numeric(Val),
                Rate=factor(Rate),
                Date=mdy(Date)) %>% 
  dplyr::mutate(Meeting=gsub("History for ","",Meeting)) %>% 
  dplyr::mutate(Meeting=gsub(" Fed meeting","",Meeting)) %>% 
  dplyr::mutate(Meeting=as.Date(parse_date_time(Meeting,"%d %b %Y"))) %>% 
  dplyr::mutate(Rate=gsub("\\)","",Rate)) %>% 
  dplyr::mutate(Rate=stringr::str_extract(Rate,"(?<=-).*")) %>% 
  dplyr::mutate(Rate=as.numeric(Rate)/100-0.25) %>% 
  # dplyr::mutate(floor_date_=floor_date(Date,unit = "month")) %>% 
  # group_by(floor_date_) %>% 
  # filter(Date==min(Date,na.rm=T)) %>% 
  # group_by(Date,Meeting) %>% 
  # filter(Val==max(Val,na.rm=T)) %>% 
  ungroup() 
  
  df_dff_bind =   df_dff %>% 
    dplyr::select(date,value) %>% 
    inner_join( temp_df %>% dplyr::select(Date) %>% dplyr::rename(date=Date)) %>% 
    dplyr::mutate(Meeting=date) %>% 
    dplyr::rename(Date=date,
                  Rate=value)
  
  temp_df %>% 
    bind_rows( df_dff_bind )
}))

meetings_with_lowest_rate = rate_probs %>% 
    filter(Val>0) %>% 
    filter(!is.na(Val)) %>% 
    arrange(Date) %>% 
    group_by(Date) %>% 
    dplyr::filter(Rate==min(Rate,na.rm=T)) %>% 
    filter(Val==max(Val,na.rm=T)) %>% 
    ungroup() %>% 
    dplyr::select(Date,Meeting) #%>% 
    # dplyr::mutate(Rate=factor(Rate))

rate_probs %>% 
  inner_join( meetings_with_lowest_rate ) %>% 
  arrange(Date) %>% 
  filter(Val>0) %>% 
  dplyr::mutate(Rate=factor(Rate)) %>% 
  ggplot(.,aes(x=Date,y=Rate,fill=Val)) +
  geom_tile() +
  scale_x_date(date_breaks = "1 month",date_labels = "%Y %b") +
  labs(x=NULL,y=NULL) +
  theme_bw() +
  theme(axis.text.x=element_text(angle=45,hjust=1)) +
  scale_fill_gradient2(low="black",high="green")
```


```r
# Rate Probabilities
filename_ = file.info(list.files("FedMeetingHistory",path = "Data",full.names = T)) %>% filter(mtime==max(mtime)) %>% row.names(.)
full_file = read.csv(file = filename_,header = FALSE)
# for (col_num in seq(2,ncol(full_file))) {
#   full_file[1,col_num] = ifelse(""==(full_file[1,col_num]),full_file[1,col_num-1],full_file[1,col_num])
# }
cols_to_loop = which(full_file %>% slice(1) %>% as.character() %like% "History for")
col_names = full_file %>% slice(1) %>% as.character()

rate_probs = do.call("rbind",lapply(seq(2,length(cols_to_loop)),function(cols_) {
  temp_df = full_file %>% 
    dplyr::select(1,c(cols_to_loop[cols_-1]:(cols_to_loop[cols_]-1))) %>% 
    slice(c(2:nrow(.)))
  colnames(temp_df) = temp_df %>% slice(c(1)) %>% as.character()
  temp_df = temp_df %>% 
    tail(-1) %>% 
    gather(Rate,Val,c(2:ncol(.))) %>% 
    dplyr::mutate(Meeting=col_names[cols_to_loop[cols_]]) %>% 
  dplyr::mutate(Val=as.numeric(Val),
                Rate=factor(Rate),
                Date=mdy(Date)) %>% 
  dplyr::mutate(Meeting=gsub("History for ","",Meeting)) %>% 
  dplyr::mutate(Meeting=gsub(" Fed meeting","",Meeting)) %>% 
  dplyr::mutate(Meeting=as.Date(parse_date_time(Meeting,"%d %b %Y"))) %>% 
  dplyr::mutate(Rate=gsub("\\)","",Rate)) %>% 
  dplyr::mutate(Rate=stringr::str_extract(Rate,"(?<=-).*")) %>% 
  dplyr::mutate(Rate=as.numeric(Rate)/100-0.25) %>% 
  dplyr::mutate(Rate=Rate*Val) %>% 
  dplyr::mutate(series_id="Futures Forecast") %>%
  dplyr::rename(date=Date) %>% 
  group_by(date,series_id) %>% 
  dplyr::summarize(value=sum(Rate,na.rm=T)) %>% 
  # dplyr::mutate(floor_date_=floor_date(Date,unit = "month")) %>% 
  # group_by(floor_date_) %>% 
  # filter(Date==min(Date,na.rm=T)) %>% 
  # group_by(Date,Meeting) %>% 
  # filter(Val==max(Val,na.rm=T)) %>% 
  ungroup() 
  
  df_dff %>% 
    dplyr::select(date,series_id,value) %>% 
    bind_rows(temp_df)

}))

p1 = rate_probs %>% 
  group_by(date,series_id) %>% 
  filter(value==max(value,na.rm=T)) %>%  
  ungroup() %>% 
  distinct() %>% 
  filter(date>=dmy("01-01-2022")) %>% 
  ggplot(.,aes(x=date,y=value,color=series_id)) +
  geom_line() +
  scale_x_date(date_breaks = "1 month",date_labels = "%Y %b") +
  labs(x=NULL,y=NULL) +
  theme_bw() +
  theme(axis.text.x=element_text(angle=45,hjust=1)) +
  scale_fill_gradient2(low="black",high="green") +
  scale_y_continuous(breaks = seq(0,6,0.25),limits = c(0,6))

p2 = rate_probs %>% 
  group_by(date,series_id) %>% 
  filter(value==max(value,na.rm=T)) %>%  
  ungroup() %>% 
  distinct() %>% 
  filter(date>=dmy("01-01-2022")) %>% 
  spread(series_id,value) %>% 
  tidyr::fill(`Futures Forecast`,.direction = "up")  %>% 
  dplyr::mutate(spread=`Futures Forecast`-DFF) %>% 
  ggplot(.,aes(x=date,y=spread,color)) +
  geom_line() +
  scale_x_date(date_breaks = "1 month",date_labels = "%Y %b",limits = c(dmy("01-03-2022",NA))) +
  labs(x=NULL,y=NULL) +
  theme_bw() +
  theme(axis.text.x=element_text(angle=45,hjust=1)) +
  scale_fill_gradient2(low="black",high="green") +
  scale_y_continuous(breaks = seq(0,3.5,0.25),limits = c(0,3.5))

grid.arrange(p1,p2)
```
